# Modeling a skydiver with quadratic drag

This notebook keeps explanation and exploration close to the model while exporting reusable calculations into a Python package. Downward velocity is negative. For a skydiver starting from rest, the solution approaches terminal velocity smoothly.

In [ ]:
#| default_exp physics

In [ ]:
#| export
import numpy as np

In [ ]:
#| export
def terminal_velocity(
    mass_kg: float,
    drag_coefficient: float,
    gravity: float = 9.81,
) -> float:
    """Return the magnitude of terminal velocity in meters per second."""
    if mass_kg <= 0 or drag_coefficient <= 0 or gravity <= 0:
        raise ValueError("mass, drag coefficient, and gravity must be positive")
    return float(np.sqrt(mass_kg * gravity / drag_coefficient))

In [ ]:
# A quick exploratory check. The test suite will preserve this assumption.
assert np.isclose(terminal_velocity(80, 0.26), 54.95, rtol=0.01)

In [ ]:
#| export
def velocity_at_time(
    time_s: float | np.ndarray,
    mass_kg: float,
    drag_coefficient: float,
    gravity: float = 9.81,
) -> float | np.ndarray:
    """Return downward velocity for a skydiver starting from rest."""
    if np.any(np.asarray(time_s) < 0):
        raise ValueError("time must be non-negative")
    limit = terminal_velocity(mass_kg, drag_coefficient, gravity)
    velocity = -limit * np.tanh(gravity * np.asarray(time_s) / limit)
    return float(velocity) if np.ndim(velocity) == 0 else velocity

In [ ]:
times = np.linspace(0, 20, 101)
velocities = velocity_at_time(times, mass_kg=80, drag_coefficient=0.26)
velocities[:5], velocities[-1]

In [ ]:
#| export
def time_to_fraction(
    fraction: float,
    mass_kg: float,
    drag_coefficient: float,
    gravity: float = 9.81,
) -> float:
    """Return seconds required to reach a fraction of terminal velocity."""
    if not 0 < fraction < 1:
        raise ValueError("fraction must be between 0 and 1")
    limit = terminal_velocity(mass_kg, drag_coefficient, gravity)
    return float(limit / gravity * np.arctanh(fraction))

In [ ]:
limit = terminal_velocity(80, 0.26)
seconds = time_to_fraction(0.99, 80, 0.26)
print(f"Terminal velocity: {limit:.2f} m/s downward")
print(f"Time to reach 99%: {seconds:.2f} s")

## Next step

Run `nbdev-export`, then inspect `skydiver/physics.py`. The package can be imported by tests, command-line programs, containers, and HPC jobs without relying on notebook execution order.